# Crawl Pipeline — Corpus Extra (239 câu)

**Trước khi chạy:**
1. Runtime → Change runtime type → **T4 GPU**
2. Thêm secrets (tab 🔑 bên trái): `GROQ_API_KEY`
3. Upload `crawl_extra_combined.jsonl` vào session

**Resume:** Nếu bị ngắt giữa chừng, chỉ cần chạy lại từ Cell 1 — pipeline tự đọc checkpoint từ Drive và tiếp tục.

**239 câu gồm:**
- 204 câu chưa có docs
- 35 câu có docs nhưng best_score < 0.9

In [ ]:
# Cell 1 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/callbot_crawl'
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
print(f'Output sẽ lưu tại: {DRIVE_OUTPUT_DIR}')

In [ ]:
# Cell 2 — Clone repo
import os

REPO_URL = 'https://github.com/enigmaticcat/legal-voice-callbot.git'
REPO_DIR = '/content/callbot'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
print('Done')

In [ ]:
# Cell 3 — Install dependencies
!pip install -q groq ddgs trafilatura sentence-transformers python-dotenv

In [ ]:
# Cell 4 — Set API key từ Colab Secrets
import os
from google.colab import userdata

os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
print('GROQ_API_KEY:', 'OK' if os.environ.get('GROQ_API_KEY') else 'MISSING')

In [ ]:
# Cell 5 — Upload crawl_extra_combined.jsonl
from google.colab import files
import shutil

uploaded = files.upload()  # chọn crawl_extra_combined.jsonl
for fname in uploaded:
    shutil.move(fname, f'/content/callbot/{fname}')
    print(f'Moved: {fname} → /content/callbot/{fname}')

In [ ]:
# Cell 6 — Kiểm tra GPU và files
import torch, os
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

has_input = os.path.exists('/content/callbot/crawl_extra_combined.jsonl')
print(f'crawl_extra_combined.jsonl: {"OK" if has_input else "MISSING — upload ở Cell 5"}')
if has_input:
    import json
    n = sum(1 for l in open('/content/callbot/crawl_extra_combined.jsonl', encoding='utf-8') if l.strip())
    print(f'Số câu cần crawl: {n}')

In [ ]:
# Cell 7 — Config paths (output + checkpoint → Drive để resume được)
import os, json
from pathlib import Path

DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/callbot_crawl'

OUTPUT_JSONL   = f'{DRIVE_OUTPUT_DIR}/new_corpus_chunks.jsonl'
FAILED_LOG     = f'{DRIVE_OUTPUT_DIR}/crawl_failed_extra.txt'
CHECKPOINT_DIR = f'{DRIVE_OUTPUT_DIR}/checkpoints_extra'  # lưu trên Drive → resume sau khi ngắt

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Báo cáo checkpoint hiện tại
ckpts = sorted(Path(CHECKPOINT_DIR).glob('checkpoint_*.json'))
if ckpts:
    saved = json.load(open(ckpts[-1]))
    print(f'♻️  Tìm thấy checkpoint: {ckpts[-1].name} — sẽ tiếp tục từ câu #{saved["next_idx"]}/239')
else:
    print('🆕 Chưa có checkpoint — bắt đầu từ đầu')

# Số docs hiện có trong output
if os.path.exists(OUTPUT_JSONL):
    existing = sum(1 for l in open(OUTPUT_JSONL, encoding='utf-8') if l.strip())
    print(f'Output hiện có: {existing} docs (sẽ append thêm)')

print(f'Output  : {OUTPUT_JSONL}')
print(f'Ckpt dir: {CHECKPOINT_DIR}')

In [ ]:
# Cell 8 — Chạy crawl pipeline
crawl_src = open('/content/callbot/crawl_pipeline.py', encoding='utf-8').read()

crawl_src = crawl_src.replace(
    'OUTPUT_JSONL        = "new_corpus_chunks.jsonl"         # append vào file cũ',
    f'OUTPUT_JSONL        = "{OUTPUT_JSONL}"'
).replace(
    'FAILED_LOG          = "crawl_failed_extra.txt"',
    f'FAILED_LOG          = "{FAILED_LOG}"'
).replace(
    'CHECKPOINT_DIR      = "checkpoints_extra"',
    f'CHECKPOINT_DIR      = "{CHECKPOINT_DIR}"'
).replace(
    'GAP_QUESTIONS_PATH  = "crawl_extra_combined.jsonl"   # 239 câu cần crawl thêm',
    'GAP_QUESTIONS_PATH  = "/content/callbot/crawl_extra_combined.jsonl"'
)

exec(crawl_src, {'__name__': '__main__'})

In [ ]:
# Cell 9 — Kiểm tra kết quả
import json, os

if os.path.exists(OUTPUT_JSONL):
    lines = open(OUTPUT_JSONL, encoding='utf-8').readlines()
    print(f'Tổng docs trong output: {len(lines)}')
    if lines:
        sample = json.loads(lines[-1])
        print(f'Latest source : {sample["source"]}')
        print(f'Latest score  : {sample["relevance_score"]}')
        print(f'Latest Q      : {sample.get("from_question","?")[:80]}')
else:
    print('Output file chưa có')